<a href="https://colab.research.google.com/github/lionfaizan999-cmyk/Cricket-Rag-chatbot/blob/main/cricket_rag_chatbot_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏏 Cricket RAG + QLoRA Chatbot — Finalized

This notebook replaces the earlier `trained40_A` notebook, which had accumulated many
duplicate/overwritten function definitions (`answer_question_v2` was redefined 5 times)
and several helper functions that were **called but never defined**
(`card_answer`, `_safe_text`, `_get_player_name_by_id`, `_normalize_id`,
`_find_player_id_from_question`, `_get_batting_for_match`, `player_id_to_name`, ...).
Because every call site was wrapped in a silent `try/except`, none of this crashed —
it just silently fell through, so the deterministic/grounded layer never actually ran.

**Finalization pass on this notebook fixed one instance of the exact same bug class:**
`deterministic_top_scorer` was referenced in the handler-routing table but never defined
anywhere — this one would have thrown a hard `NameError` (not a silent failure) the moment
the master-engine cell ran, since it's used as a raw dict value. It's now implemented,
mirroring `deterministic_top_wicket_taker`. This notebook also drops several disconnected
scratch/experiment cells that used to sit between the pipeline and the chat UI (toy
fine-tuning runs on hardcoded examples, a duplicate mini-RAG demo) — they weren't used by
`answer_question()` and some of them reused global variable names (`base_model`, `model`,
`trainer`) that the real pipeline depends on, so removing them also removes a real risk of
silent corruption if the notebook is run top-to-bottom in one session.

**Architecture** (matches the plan already agreed on):

```
User Question
    │
    ├─► Cricket-domain guard
    ├─► Intent detection (player_runs, toss_winner, venue, ...)
    │
    ├─► STRUCTURED / DETERMINISTIC LAYER (always tried first for factual questions)
    │     • player runs / wickets, top scorer / top wicket-taker (with
    │       optional opponent + year + "final" scoping)
    │     • match winner / toss / venue / date / final teams / runner-up
    │     • match summary / player of the match
    │
    ├─► If the question is one where a wrong number would be worse than "I don't know"
    │   (player_runs, player_wickets, team_runs, winner, toss, ...) and the structured
    │   layer found nothing → return "I could not find a reliable answer", never guess.
    │
    └─► Otherwise (open-ended/descriptive questions) → semantic retrieval (RAG)
          over `cricket_documents.pkl` / `cricket_embeddings.npy`, context handed to
          the trained QLoRA-adapted Qwen model for a grounded, context-only answer.
```

**Data source:** everything is downloaded from Hugging Face (CSVs + RAG assets from the `cricket-odi-data` dataset repo, adapter from the
`qlora-cricket-adapter-grounded` model repo; see the config cell) — no Google Drive needed.

Existing assets are **reused, not retrained**: `cricket_documents.pkl`,
`cricket_embeddings.npy`, the ODI CSVs, and a trained QLoRA adapter. The adapter used is
`qlora_cricket_adapter_grounded`.

Run the cells in order. After the pip-install cell, restart the Colab runtime once,
then run from the top.


In [1]:
%pip install -q --upgrade \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "scikit-learn==1.6.1" \
    "sentence-transformers==4.1.0" \
    "transformers==4.57.6" \
    "accelerate" \
    "bitsandbytes>=0.46.1" \
    "peft" \
    "datasets" \
    "gradio" \
    "trl"

# NumPy, SciPy, Scikit-learn, Sentence-Transformers (for embedding)
# Transformers (for Qwen)
# PEFT (for the QLoRA adapter)
# Gradio (for the chat UI)
# Datasets (for fine-tuning)
# TRL (for SFTTrainer fine-tuning)

print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ALL IMPORTS — keep imports only in this cell

import os
import re
import pickle
import warnings
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from huggingface_hub import hf_hub_download, snapshot_download, list_repo_files

import gradio as gr

warnings.filterwarnings("ignore")
print("✅ Imports loaded.")

✅ Imports loaded.


In [2]:
# ============================================
# DATA + MODEL SOURCE: Hugging Face Hub
# ============================================
import os
import torch
from huggingface_hub import hf_hub_download, snapshot_download, list_repo_files

# CSVs + RAG assets  -> dataset repo
# QLoRA adapter      -> model repo
# Koi Google Drive / local file zaroori nahi.

DATA_REPO_ID = "Faizan93Ali/cricket-odi-data"                    # dataset repo
ADAPTER_REPO_ID = "Faizan93Ali/qlora-cricket-adapter-grounded"   # model repo
HF_TOKEN = None   # repos private hon to yahan HF token string daalein

# Downloaded files yahan local Colab disk par save hongi
PROJECT_DIR = "/content/Cricket_RAG"
os.makedirs(PROJECT_DIR, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Sirf grounded adapter HF par upload hai
ADAPTER_CHOICE = "grounded"

# ---- Dataset files: repo mein jahan bhi hon (root ya subfolder), naam se dhoond kar download ----
_data_files = list_repo_files(DATA_REPO_ID, repo_type="dataset", token=HF_TOKEN)

def _hf_download_file(basename: str, required: bool = True):
    found = [f for f in _data_files if os.path.basename(f) == basename]
    if not found:
        if required:
            raise FileNotFoundError(
                f"❌ '{basename}' not found in {DATA_REPO_ID}.\nFiles in repo: {_data_files}"
            )
        print(f"⚠️ Optional file '{basename}' not in {DATA_REPO_ID} — will be built from CSVs.")
        return os.path.join(PROJECT_DIR, basename)
    return hf_hub_download(
        repo_id=DATA_REPO_ID, filename=found[0], repo_type="dataset",
        token=HF_TOKEN, local_dir=PROJECT_DIR,
    )

# Required: structured CSVs
MATCHES_PATH = _hf_download_file("odi_Matches_Data.csv")
BATTING_PATH = _hf_download_file("odi_Batting_Card.csv")
BOWLING_PATH = _hf_download_file("odi_Bowling_Card.csv")
FOW_PATH = _hf_download_file("odi_Fow_Card.csv")
PARTNERSHIP_PATH = _hf_download_file("odi_Partnership_Card.csv")
PLAYERS_PATH = _hf_download_file("players_info.csv")

# Optional: pre-built RAG assets (na hon to cell 7 CSVs se khud bana leta hai)
DOCUMENTS_PATH = _hf_download_file("cricket_documents.pkl", required=False)
EMBEDDINGS_PATH = _hf_download_file("cricket_embeddings.npy", required=False)

# ---- QLoRA adapter (model repo) ----
ADAPTER_PATH = os.path.join(PROJECT_DIR, "qlora_cricket_adapter_grounded")
try:
    snapshot_download(
        repo_id=ADAPTER_REPO_ID, repo_type="model", token=HF_TOKEN,
        local_dir=ADAPTER_PATH,
    )
    # adapter_config.json jis folder mein ho wahi adapter folder hai
    _found_adapter = None
    for _root, _dirs, _files in os.walk(ADAPTER_PATH):
        if "adapter_config.json" in _files:
            _found_adapter = _root
            break
    if _found_adapter is None:
        raise FileNotFoundError("adapter_config.json not found in downloaded adapter repo")
    ADAPTER_PATH = _found_adapter
except Exception as e:
    print(f"⚠️ Adapter download failed ({type(e).__name__}): {e}")
    ADAPTER_PATH = os.path.join(PROJECT_DIR, "adapter_not_found")  # cell 8 base model use karega

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Data source (dataset):", DATA_REPO_ID)
print("Adapter source (model):", ADAPTER_REPO_ID)
print("Local directory:", PROJECT_DIR)
print("Adapter path:", ADAPTER_PATH)

odi_Matches_Data.csv: 0.00B [00:00, ?B/s]

odi_Batting_Card.csv: 0.00B [00:00, ?B/s]

odi_Bowling_Card.csv: 0.00B [00:00, ?B/s]

odi_Fow_Card.csv: 0.00B [00:00, ?B/s]

odi_Partnership_Card.csv: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Device: cuda
Data source (dataset): Faizan93Ali/cricket-odi-data
Adapter source (model): Faizan93Ali/qlora-cricket-adapter-grounded
Local directory: /content/Cricket_RAG
Adapter path: /content/Cricket_RAG/qlora_cricket_adapter_grounded


In [3]:
if "MATCHES_PATH" not in globals():
    raise RuntimeError(
        "❌ Cell 3 (Hugging Face download) pehle successfully run nahi hui — "
        "MATCHES_PATH abhi define nahi hai. Cell 2 aur cell 3 run karein, phir yeh cell."
    )

def load_csv(path: str, required_columns=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"\n❌ DATASET FILE NOT FOUND\nPath: {path}")

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    print(f"✅ {os.path.basename(path)} -> {df.shape}")

    if required_columns:
        missing = [col for col in required_columns if col not in df.columns]
        if missing:
            raise ValueError(
                f"\n❌ WRONG DATASET LOADED\nFile: {os.path.basename(path)}\n"
                f"Missing columns: {missing}\nActual columns: {df.columns.tolist()}"
            )
    return df


matches = load_csv(MATCHES_PATH, required_columns=[
    "Match ID", "Match Name", "Match Date", "Team1 Name", "Team2 Name"
])
batting = load_csv(BATTING_PATH, required_columns=["Match ID", "batsman", "runs"])
bowling = load_csv(BOWLING_PATH, required_columns=["Match ID"])
fow = load_csv(FOW_PATH, required_columns=["Match ID"])
partnership = load_csv(PARTNERSHIP_PATH, required_columns=["Match ID"])
players = load_csv(PLAYERS_PATH, required_columns=["player_id", "player_name"])

print("\n" + "=" * 70)
print("DATASET VALIDATION PASSED")
print("=" * 70)
print("matches    :", matches.shape)
print("batting    :", batting.shape)
print("bowling    :", bowling.shape)
print("fow        :", fow.shape)
print("partnership:", partnership.shape)
print("players    :", players.shape)

✅ odi_Matches_Data.csv -> (4745, 33)
✅ odi_Batting_Card.csv -> (103225, 13)
✅ odi_Bowling_Card.csv -> (56514, 16)
✅ odi_Fow_Card.csv -> (67963, 7)
✅ odi_Partnership_Card.csv -> (73826, 13)
✅ players_info.csv -> (6701, 11)

DATASET VALIDATION PASSED
matches    : (4745, 33)
batting    : (103225, 13)
bowling    : (56514, 16)
fow        : (67963, 7)
partnership: (73826, 13)
players    : (6701, 11)


In [4]:
def normalize_columns(df):
    if df is None:
        return None
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def add_normalized_date(df):
    if df is None or "Match Date" not in df.columns:
        return df
    out = df.copy()
    out["_date"] = pd.to_datetime(out["Match Date"], errors="coerce")
    out["_date_str"] = out["_date"].dt.strftime("%Y-%m-%d")
    out["_year"] = out["_date"].dt.year
    return out


matches = add_normalized_date(normalize_columns(matches))
batting = add_normalized_date(normalize_columns(batting))
bowling = add_normalized_date(normalize_columns(bowling))
fow = add_normalized_date(normalize_columns(fow))
partnership = add_normalized_date(normalize_columns(partnership))
players = normalize_columns(players)

print("✅ Columns normalized, date fields added.")
print("Match columns:", matches.columns.tolist())

✅ Columns normalized, date fields added.
Match columns: ['ODI Match No', 'Match ID', 'Match Name', 'Series ID', 'Series Name', 'Match Date', 'Match Format', 'Team1 ID', 'Team1 Name', 'Team1 Captain', 'Team1 Runs Scored', 'Team1 Wickets Fell', 'Team1 Extras Rec', 'Team2 ID', 'Team2 Name', 'Team2 Captain', 'Team2 Runs Scored', 'Team2 Wickets Fell', 'Team2 Extras Rec', 'Match Venue (Stadium)', 'Match Venue (City)', 'Match Venue (Country)', 'Umpire 1', 'Umpire 2', 'Match Referee', 'Toss Winner', 'Toss Winner Choice', 'Match Winner', 'Match Result Text', 'MOM Player', 'Team1 Playing 11', 'Team2 Playing 11', 'Debut Players', '_date', '_date_str', '_year']


## RAG documents & embeddings

Loads your existing `cricket_documents.pkl` / `cricket_embeddings.npy` if present.
If either is missing, it builds them from the structured CSVs (one document per
match, batting entry, and bowling entry) so the notebook is still runnable from a
fresh session — set `REBUILD_RAG_ASSETS = True` below to force a rebuild
even if the files already exist.

In [5]:
REBUILD_RAG_ASSETS = False  # set True to force regeneration even if files exist

documents = None
embeddings = None

def _row_to_document(prefix, row, columns):
    parts = [f"{col}: {row.get(col)}" for col in columns if pd.notna(row.get(col))]
    return f"{prefix} | " + " | ".join(parts)

def build_documents():
    docs = []
    match_cols = [c for c in matches.columns if not str(c).startswith("_")]
    for _, row in matches.iterrows():
        docs.append(_row_to_document("MATCH", row, match_cols))

    batting_cols = [c for c in batting.columns if not str(c).startswith("_")]
    for _, row in batting.iterrows():
        docs.append(_row_to_document("BATTING", row, batting_cols))

    bowling_cols = [c for c in bowling.columns if not str(c).startswith("_")]
    for _, row in bowling.iterrows():
        docs.append(_row_to_document("BOWLING", row, bowling_cols))

    return docs


need_build = (
    REBUILD_RAG_ASSETS
    or not os.path.exists(DOCUMENTS_PATH)
    or not os.path.exists(EMBEDDINGS_PATH)
)

if not need_build:
    with open(DOCUMENTS_PATH, "rb") as f:
        documents = pickle.load(f)
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"✅ Loaded existing RAG assets: {len(documents)} documents, "
          f"embeddings {embeddings.shape}")
else:
    print("⚠️ RAG assets missing (or rebuild requested) — building from CSVs...")
    documents = build_documents()
    _temp_embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
    embeddings = _temp_embedder.encode(
        documents, convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=True, batch_size=64,
    )
    os.makedirs(os.path.dirname(DOCUMENTS_PATH), exist_ok=True)
    with open(DOCUMENTS_PATH, "wb") as f:
        pickle.dump(documents, f)
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"✅ Built and saved {len(documents)} documents, embeddings {embeddings.shape}")

assert embeddings.shape[0] == len(documents), "Embeddings/documents count mismatch!"


✅ Loaded existing RAG assets: 4745 documents, embeddings (4745, 384)


In [6]:
print("Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
print("✅ Embedding model loaded.")

print("Loading base Qwen model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # matches the grounded adapter's training config
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

if os.path.exists(ADAPTER_PATH):
    print("Loading trained QLoRA adapter from:", ADAPTER_PATH)
    generator = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    print(f"✅ Trained QLoRA adapter loaded ({ADAPTER_CHOICE}, reused, not retrained).")
else:
    print("⚠️ No trained adapter found at ADAPTER_PATH — using the base Qwen model as-is.")
    generator = base_model

generator.eval()
print("🏏 Models ready.")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading trained QLoRA adapter from: /content/Cricket_RAG/qlora_cricket_adapter_grounded
✅ Trained QLoRA adapter loaded (grounded, reused, not retrained).
🏏 Models ready.


## Text utilities & intent detection

In [7]:
def clean_text(value) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()

_safe_text = clean_text  # alias kept for readability in older call sites

def _normalize_text(value) -> str:
    return clean_text(value).lower()

def normalize_team(value) -> str:
    return re.sub(r"\s+", " ", clean_text(value).lower()).strip()

def _normalize_entity_for_match(value) -> str:
    if value is None:
        return ""
    value = clean_text(value).lower()
    return re.sub(r"[^a-z0-9]+", "", value)

_safe_normalize_entity = _normalize_entity_for_match

def _normalize_id(value):
    """Normalize an ID (player id / match id) so int/float/str all compare equal."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return None
    try:
        f = float(text)
        if f.is_integer():
            return str(int(f))
        return str(f)
    except ValueError:
        return text.lower()

def extract_date(query: str) -> Optional[str]:
    q = clean_text(query).lower()
    m = re.search(r"\b((?:19|20)\d{2})[-/](\d{1,2})[-/](\d{1,2})\b", q)
    if m:
        try:
            return pd.Timestamp(year=int(m.group(1)), month=int(m.group(2)), day=int(m.group(3))).strftime("%Y-%m-%d")
        except Exception:
            return None
    return None

def extract_year(query: str) -> Optional[int]:
    m = re.search(r"\b((?:19|20)\d{2})\b", clean_text(query).lower())
    return int(m.group(1)) if m else None

def is_final_question(question: str) -> bool:
    q = clean_text(question).lower()
    final_terms = ["final", "runner-up", "runner up", "runnerup", "champion", "champions",
                   "won the cup", "world cup winner", "world cup champion"]
    if any(term in q for term in final_terms):
        return True
    # "won the world cup" ke beech mein saal aa sakta hai, e.g. "won the 2023 world cup"
    if re.search(r"won the (?:\d{4}\s+)?(?:world cup|cup)\b", q):
        return True
    return False

def is_cricket_question(question: str) -> bool:
    q = clean_text(question).lower()
    cricket_terms = ["cricket", "match", "world cup", "cup", "odi", "t20", "test", "toss",
                      "run", "runs", "wicket", "wickets", "batting", "bowling", "batsman",
                      "batter", "bowler", "final", "semi final", "quarter final", "team",
                      "player", "score", "scored", "venue", "stadium", "ground", "partnership",
                      "dismissed", "fall of wicket", "winner", "won", "lost", "loser",
                      "runner-up", "runner up", "champion", "captain", "man of the match",
                      "player of the match", "summary", "result", "margin"]
    return any(term in q for term in cricket_terms) or bool(re.search(r"\b(?:19|20)\d{2}\b", q))


def detect_intent(question: str) -> str:
    q = clean_text(question).lower()

    if any(p in q for p in ["give me the summary", "give me a summary", "summarize", "summarise",
                             "match summary", "summary of"]):
        return "match_summary"

    if (any(p in q for p in ["what was the result", "match result", "what was the match result",
                              "how did the match end", "how did the final end", "how did the game end",
                              "how did the match finish", "how did the final finish"])
        or ("how did" in q and any(w in q for w in ["match", "final", "game"])
            and any(w in q for w in ["end", "ended", "finish", "finished"]))):
        return "match_result"

    if (any(p in q for p in ["winning margin", "win margin", "margin of victory",
                              "by how many runs did", "by how many wickets did", "how many runs did"])
        and any(p in q for p in ["win", "won", "victory", "win by"])):
        return "winning_margin"

    if any(p in q for p in ["what were the scores", "match scores", "scores in the match",
                             "scorecard result", "scorecard", "both teams score", "both teams scores"]):
        return "match_scores"

    if (any(p in q for p in ["toss decision", "toss choice", "what did the toss winner choose",
                              "what did toss winner choose", "what did the winner choose",
                              "what did the winner of the toss choose", "what did the toss winner decide",
                              "what did toss winner decide", "what was the toss decision",
                              "what was the toss choice", "choose to bat", "choose to bowl",
                              "chose to bat", "chose to bowl", "elected to bat", "elected to bowl",
                              "bat or bowl", "bat first or bowl first", "batted or bowled"])
        or ("after winning the toss" in q and any(w in q for w in ["choose", "chose", "bat", "bowl"]))
        or ("did" in q and any(p in q for p in ["choose to bat", "choose to bowl", "elect to bat",
                                                  "elect to bowl", "elected to bat", "elected to bowl"]))):
        return "toss_decision"

    if any(p in q for p in ["who won the toss", "who won toss", "which team won the toss",
                             "which team won toss", "toss winner", "won the toss"]):
        return "toss_winner"

    if any(p in q for p in ["who lost", "which team lost", "loser", "losing team", "who was defeated",
                             "which team was defeated", "who lost the match", "who lost the final",
                             "who lost the game", "which team lost the match", "which team lost the final"]):
        return "loser"

    if any(p in q for p in ["runner-up", "runner up", "runnerup"]):
        return "runner_up"

    if "final" in q and any(p in q for p in ["defeat", "defeated", "beat", "beaten", "lost to",
                                              "opponent", "which team did"]):
        return "final_opponent"

    if "final" in q and any(p in q for p in ["teams", "which teams", "who played", "teams played"]):
        return "final_teams"

    fow_patterns = ["fall of wicket", "fall of wickets", "wicket fell", "wicket fall", "first wicket",
                     "second wicket", "third wicket", "fourth wicket", "fifth wicket", "sixth wicket",
                     "seventh wicket", "eighth wicket", "ninth wicket", "tenth wicket", "last wicket",
                     "when did the wicket fall", "score when the wicket fell", "who was dismissed",
                     "who was dismissed at", "who got dismissed", "dismissed at the", "dismissed on",
                     "dismissed in"]
    if any(p in q for p in fow_patterns) or re.search(r"\bwicket\s*(?:number|no\.?|#)?\s*\d{1,2}\b", q) or re.search(r"\b\d{1,2}(?:st|nd|rd|th)\s+wicket\b", q):
        return "fow"

    if (any(p in q for p in ["highest team score", "highest team total", "maximum team score",
                              "maximum team total", "highest score by a team", "highest total by a team",
                              "most runs by a team"])
        or (any(p in q for p in ["highest", "maximum", "max"]) and any(p in q for p in ["team score", "team total"]))):
        return "highest_team_runs"

    if any(p in q for p in ["most runs", "top scorer", "top run scorer", "highest scorer",
                             "highest scoring batsman", "highest scoring batter", "who scored the most",
                             "who made the most runs", "who scored the highest runs",
                             "scored the highest runs", "made the highest runs"]):
        return "top_scorer"

    if any(p in q for p in ["highest individual score", "highest individual runs",
                             "highest score by a player", "best individual score"]):
        return "highest_score"

    if any(p in q for p in ["how many runs did", "how many runs has", "how many runs scored by",
                             "how many did"]):
        if question_mentions_player(q):
            return "player_runs"

    top_wicket_patterns = ["top wicket taker", "top wicket-taker", "highest wicket taker",
                            "highest wicket-taker", "leading wicket taker", "leading wicket-taker",
                            "most wickets", "who took the most wickets", "who took most wickets",
                            "who got the most wickets", "who got most wickets", "best bowling figures"]
    if any(p in q for p in top_wicket_patterns):
        return "top_wicket_taker"

    player_wicket_patterns = [
        r"\bhow many wickets did\s+.+?\s+(?:take|get)\b",
        r"\bhow many wickets has\s+.+?\s+(?:taken|got)\b",
        r"\bhow many wickets were taken by\s+.+?\b",
        r"\bwickets did\s+.+?\s+(?:take|get)\b",
        r"\bwickets were taken by\s+.+?\b",
    ]
    if any(re.search(pattern, q) for pattern in player_wicket_patterns):
        return "player_wickets"

    if any(p in q for p in ["wicket", "wickets", "bowled", "bowling"]):
        return "wickets"

    if any(p in q for p in ["player of the match", "player of match", "man of the match", "mom", "who was mom"]):
        return "player_of_match"

    if "partnership" in q:
        return "partnership"

    if any(p in q for p in ["venue", "stadium", "ground", "where was", "played at"]):
        return "venue"

    if any(p in q for p in ["when was", "when did", "what date", "match date", "date of", "played on"]):
        return "date"

    if any(p in q for p in ["who won", "who was the winner", "which team won", "winner of", "winning team"]):
        return "winner"

    team_score_patterns = [
        r"\b[\w][\w\s&.-]*'s\s+(?:score|total|runs)\b",
        r"\b(?:score|total)\s+(?:for|of)\s+[\w][\w\s&.-]*",
        r"\b[\w][\w\s&.-]*\s+score\b",
        r"\b[\w][\w\s&.-]*\s+scored\b",
        r"\bhow many runs did\b",
        r"\bhow many runs has\b",
        r"\bhow many runs\b",
        r"\bwhat did the team score\b",
        r"\bwhat was the team score\b",
        r"\bwhat was the team total\b",
        r"\bwhat was the score\b",
        r"\bteam score\b",
        r"\bteam total\b",
        r"\bruns scored\b",
    ]
    if any(re.search(pattern, q) for pattern in team_score_patterns):
        if not question_mentions_player(q):
            return "team_runs"

    return "general"

print("✅ Text utilities and intent detection loaded.")

✅ Text utilities and intent detection loaded.


## Player / team lookup & entity extraction

`matches`, `batting`, `bowling`, `fow`, `partnership`, `players` were already loaded
above — the cell below just points the pipeline's internal globals at them and builds
the `player_id ↔ player_name` lookup once.

In [8]:

# matches / batting / bowling / fow / partnership / players / documents /
# embeddings / embedding_model / tokenizer / generator were already created
# by the cells above; nothing to reassign here.

player_id_to_name: Dict[str, str] = {}
name_to_player_id: Dict[str, str] = {}

def build_player_lookup():
    """Build id<->name maps from the players dataframe. Call after loading data."""
    global player_id_to_name, name_to_player_id
    player_id_to_name = {}
    name_to_player_id = {}
    if players is None or players.empty:
        return
    if "player_id" not in players.columns or "player_name" not in players.columns:
        return
    for _, row in players[["player_id", "player_name"]].dropna().drop_duplicates().iterrows():
        pid = _normalize_id(row["player_id"])
        name = clean_text(row["player_name"])
        if pid is not None and name:
            player_id_to_name[pid] = name
            name_to_player_id.setdefault(name.lower(), pid)

def _get_player_name_by_id(player_id) -> Optional[str]:
    pid = _normalize_id(player_id)
    if pid is None:
        return None
    return player_id_to_name.get(pid)


def question_mentions_player(question: str) -> bool:
    if players is None or players.empty or "player_name" not in players.columns:
        return False
    q = clean_text(question).lower()
    names = players["player_name"].dropna().astype(str).str.strip().drop_duplicates().tolist()
    names.sort(key=len, reverse=True)
    for name in names:
        if not name:
            continue
        pattern = r"(?<!\w)" + re.escape(name.lower()) + r"(?!\w)"
        if re.search(pattern, q):
            return True
    return False

def question_mentions_team(question: str) -> bool:
    q = clean_text(question).lower()
    teams = []
    try:
        if matches is not None and not matches.empty:
            for col in ["Team1 Name", "Team2 Name", "Match Winner", "Toss Winner"]:
                if col in matches.columns:
                    teams.extend(matches[col].dropna().astype(str).str.strip().tolist())
    except Exception:
        pass
    teams = list(dict.fromkeys(t for t in teams if t))
    teams.sort(key=len, reverse=True)
    for team in teams:
        pattern = r"(?<!\w)" + re.escape(team.lower()) + r"(?!\w)"
        if re.search(pattern, q):
            return True
    return False


TEAM_ALIASES = {
    "pak": "pakistan", "ind": "india", "aus": "australia", "eng": "england",
    "nz": "newzealand", "sa": "southafrica", "wi": "westindies", "sl": "srilanka",
    "ban": "bangladesh", "zim": "zimbabwe", "ire": "ireland", "ned": "netherlands",
    "uae": "unitedarabemirates",
}

def available_teams() -> List[str]:
    if matches is None or matches.empty:
        return []
    teams_found = set()
    for col in ["Team1 Name", "Team2 Name"]:
        if col not in matches.columns:
            continue
        for value in matches[col].dropna():
            team = clean_text(value)
            if team:
                teams_found.add(team)
    return sorted(teams_found, key=lambda x: len(str(x)), reverse=True)

def _team_matches_question(normalized_team: str, normalized_question: str) -> bool:
    if not normalized_team or len(normalized_team) < 4:
        return False
    return normalized_team in normalized_question

def extract_teams(question: str) -> List[str]:
    if matches is None or matches.empty:
        return []
    q = clean_text(question)
    if not q:
        return []
    normalized_question = _normalize_entity_for_match(q)
    team_names = available_teams()
    if not team_names:
        return []
    normalized_to_original = {}
    for team in team_names:
        nt = _normalize_entity_for_match(team)
        if nt and nt not in normalized_to_original:
            normalized_to_original[nt] = team
    expanded_question = normalized_question
    q_lower = q.lower()
    for alias, canonical in TEAM_ALIASES.items():
        pattern = r"(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])"
        if re.search(pattern, q_lower):
            expanded_question += canonical
    found = []
    sorted_teams = sorted(normalized_to_original.items(), key=lambda x: len(x[0]), reverse=True)
    for nt, original in sorted_teams:
        if _team_matches_question(nt, expanded_question):
            found.append(original)
    result, seen = [], set()
    for team in found:
        key = _normalize_entity_for_match(team)
        if key not in seen:
            seen.add(key)
            result.append(team)
    return result[:2]

def _extract_opponent_from_question(question: str) -> Optional[str]:
    """Prefer an exact dataset team name; fall back to regex after against/vs/versus."""
    q = clean_text(question)
    if not q:
        return None
    q_lower = q.lower()
    for team in available_teams():
        nt = _normalize_entity_for_match(team)
        if len(nt) >= 4 and nt in _normalize_entity_for_match(q):
            if re.search(r"\b(against|vs\.?|versus)\b", q_lower):
                return team
    patterns = [
        r"\bagainst\s+(.+?)(?:\s+in\s+\d{4}|\s+in\s+this\s+match|\?|$)",
        r"\bvs\.?\s+(.+?)(?:\s+in\s+\d{4}|\s+in\s+this\s+match|\?|$)",
        r"\bversus\s+(.+?)(?:\s+in\s+\d{4}|\s+in\s+this\s+match|\?|$)",
    ]
    for pattern in patterns:
        m = re.search(pattern, q, flags=re.IGNORECASE)
        if m:
            opponent = m.group(1).strip()
            opponent = re.sub(r"\b(the|team|side)\b$", "", opponent, flags=re.IGNORECASE).strip()
            if opponent:
                return opponent
    return None

def _find_player_id_from_question(question: str) -> Optional[str]:
    """Resolve a player mentioned in the question to a player_id, via full-name then surname match."""
    q = str(question).lower()
    q = re.sub(r"\b([a-z]+)'s\b", r"\1", q)

    full_matches = []
    for pid, name in player_id_to_name.items():
        name_clean = str(name).lower().strip()
        if name_clean and name_clean in q:
            full_matches.append((pid, len(name_clean)))
    if full_matches:
        full_matches.sort(key=lambda x: x[1], reverse=True)
        return full_matches[0][0]

    surname_matches = []
    for pid, name in player_id_to_name.items():
        parts = str(name).lower().split()
        if not parts:
            continue
        surname = parts[-1]
        if len(surname) >= 4 and re.search(rf"\b{re.escape(surname)}\b", q):
            surname_matches.append(pid)
    if len(surname_matches) == 1:
        return surname_matches[0]
    return None

build_player_lookup()
print(f"✅ Player lookup built: {len(player_id_to_name)} players.")

TEAM_NAMES = available_teams()
print(f"✅ Teams initialized: {len(TEAM_NAMES)}")
if TEAM_NAMES:
    print("Sample teams:", TEAM_NAMES[:10])

✅ Player lookup built: 6701 players.
✅ Teams initialized: 29
Sample teams: ['South Africa', 'ICC World XI', 'East Africa', 'Netherlands', 'New Zealand', 'Afghanistan', 'West Indies', 'Bangladesh', 'Hong Kong', 'Sri Lanka']


## Match resolution (finding the right row in `matches` for a question)

In [9]:

def _apply_world_cup_filter(df, question):
    """World Cup filter — restricts to Qualifier/League series only when
    explicitly asked, otherwise excludes them so the main tournament wins."""
    if "Series Name" not in df.columns:
        return df
    series = df["Series Name"].astype(str)
    mask = series.str.contains("world cup", case=False, na=False)
    q = clean_text(question).lower()

    if "qualifier" in q:
        mask &= series.str.contains("qualifier", case=False, na=False)
    elif "league" in q:
        mask &= series.str.contains("league", case=False, na=False)
    else:
        mask &= ~series.str.contains("qualifier|league", case=False, na=False, regex=True)

    return df[mask]


def _get_scoped_match_ids(question: str):
    """
    Return the set of normalized Match IDs that satisfy any explicit
    scope mentioned in the question (date / year / opponent / two teams /
    world cup / final). Returns None when the question carries no such
    scope at all (caller should then use ALL of a player's data).
    """
    if matches is None or matches.empty:
        return None

    df = matches.copy()
    applied = False

    exact_date = extract_date(question)
    if exact_date and "_date_str" in df.columns:
        df = df[df["_date_str"].astype(str).eq(str(exact_date))]
        applied = True
    else:
        year = extract_year(question)
        if year is not None:
            if "_year" in df.columns:
                years = pd.to_numeric(df["_year"], errors="coerce")
                df = df[years.eq(year)]
                applied = True
            elif "Match Date" in df.columns:
                dates = pd.to_datetime(df["Match Date"], errors="coerce")
                df = df[dates.dt.year.eq(year)]
                applied = True

    if "Team1 Name" in df.columns and "Team2 Name" in df.columns:
        opponent = _extract_opponent_from_question(question)
        if opponent:
            opp_n = _normalize_entity_for_match(opponent)
            t1 = df["Team1 Name"].map(_normalize_entity_for_match)
            t2 = df["Team2 Name"].map(_normalize_entity_for_match)
            df = df[t1.eq(opp_n) | t2.eq(opp_n)]
            applied = True
        else:
            teams = extract_teams(question)
            for team in teams:
                team_n = _normalize_entity_for_match(team)
                t1 = df["Team1 Name"].map(_normalize_entity_for_match)
                t2 = df["Team2 Name"].map(_normalize_entity_for_match)
                df = df[t1.eq(team_n) | t2.eq(team_n)]
                applied = True

    if "world cup" in clean_text(question).lower():
        df = _apply_world_cup_filter(df, question)
        applied = True

    if is_final_question(question) and "Match Name" in df.columns:
        names = df["Match Name"].fillna("").astype(str).str.lower()
        final_mask = names.str.contains(r"\bfinal\b", regex=True, na=False)
        non_final_mask = names.str.contains(
            r"\b(?:quarter|semi|qualifier|eliminator)\s*[- ]?\s*final\b", regex=True, na=False
        )
        df = df[final_mask & ~non_final_mask]
        applied = True

    if not applied:
        return None
    if "Match ID" not in df.columns:
        return set()
    return {mid for mid in (_normalize_id(v) for v in df["Match ID"]) if mid is not None}


def _find_match_for_question(question):
    """Resolve the single best-matching row of `matches` for a question."""
    if matches is None or matches.empty:
        return None

    q = _normalize_text(question)
    if not q:
        return None

    df = matches.copy()

    exact_date = extract_date(q)
    if exact_date and "_date_str" in df.columns:
        df = df[df["_date_str"].astype(str).eq(str(exact_date))].copy()
    else:
        year = extract_year(q)
        if year is not None:
            if "_year" in df.columns:
                year_values = pd.to_numeric(df["_year"], errors="coerce")
                df = df[year_values.eq(int(year))].copy()
            elif "Match Date" in df.columns:
                dates = pd.to_datetime(df["Match Date"], errors="coerce")
                df = df[dates.dt.year.eq(int(year))].copy()

    if df.empty:
        return None

    if "world cup" in q and "Series Name" in df.columns:
        df = _apply_world_cup_filter(df, q)
    if df.empty:
        return None

    is_final_query = bool(re.search(r"\bfinal\b", q)) or any(
        p in q for p in ["runner-up", "runner up", "runnerup"]
    )
    if is_final_query and "Match Name" in df.columns:
        match_names = df["Match Name"].fillna("").astype(str).str.lower().str.strip()
        final_mask = match_names.str.contains(r"\bfinal\b", regex=True, na=False)
        non_final_mask = match_names.str.contains(
            r"\b(?:quarter|semi|qualifier|eliminator)\s*[- ]?\s*final\b", regex=True, na=False
        )
        df = df[final_mask & ~non_final_mask].copy()
    if df.empty:
        return None

    mentioned_teams = extract_teams(q)
    if mentioned_teams:
        if "Team1 Name" not in df.columns or "Team2 Name" not in df.columns:
            return None
        for team in mentioned_teams:
            team_n = _safe_normalize_entity(team)
            if not team_n:
                continue
            t1 = df["Team1 Name"].astype(str).map(_safe_normalize_entity)
            t2 = df["Team2 Name"].astype(str).map(_safe_normalize_entity)
            mask = t1.str.contains(team_n, regex=False, na=False) | t2.str.contains(team_n, regex=False, na=False)
            df = df[mask].copy()
            if df.empty:
                return None

    if len(df) == 1:
        return df.iloc[0]

    q_tokens = set(re.findall(r"[a-z0-9]+", q.lower()))
    if "Match Name" in df.columns:
        def score_row(row):
            text = " ".join([
                str(row.get("Match Name", "")), str(row.get("Series Name", "")),
                str(row.get("Team1 Name", "")), str(row.get("Team2 Name", "")),
            ]).lower()
            tokens = set(re.findall(r"[a-z0-9]+", text))
            return len(q_tokens.intersection(tokens))
        df = df.copy()
        df["_match_score"] = df.apply(score_row, axis=1)
        df = df.sort_values("_match_score", ascending=False)
        if not df.empty:
            return df.iloc[0]

    return None


def _get_batting_for_match(match_id):
    if batting is None or batting.empty or "Match ID" not in batting.columns:
        return pd.DataFrame()
    mid = _normalize_id(match_id)
    if mid is None:
        return pd.DataFrame()
    ids = batting["Match ID"].map(_normalize_id)
    return batting[ids.eq(mid)].copy()


def _get_bowling_for_match(match_id):
    if bowling is None or bowling.empty or "Match ID" not in bowling.columns:
        return pd.DataFrame()
    mid = _normalize_id(match_id)
    if mid is None:
        return pd.DataFrame()
    ids = bowling["Match ID"].map(_normalize_id)
    return bowling[ids.eq(mid)].copy()

print("✅ Match resolution loaded (world cup qualifier/league fix merged).")


✅ Match resolution loaded (world cup qualifier/league fix merged).


In [10]:
def metadata_search(question: str) -> pd.DataFrame:
    """Filter the `matches` dataframe down to the rows that satisfy the
    scope implied by the question (date / year / opponent / two teams /
    world cup / final). Reuses the same scoping logic as
    `_get_scoped_match_ids` so match resolution stays consistent across
    the whole pipeline.
    Returns an empty DataFrame if no scope could be determined, or if
    `matches` isn't loaded.
    """
    if matches is None or matches.empty:
        return pd.DataFrame()

    scoped_ids = _get_scoped_match_ids(question)
    if scoped_ids is None:
        return pd.DataFrame()
    if "Match ID" not in matches.columns:
        return pd.DataFrame()

    ids = matches["Match ID"].map(_normalize_id)
    return matches[ids.isin(scoped_ids)].copy()

print("✅ metadata_search loaded.")

✅ metadata_search loaded.


In [11]:

_DESCRIPTIVE_PATTERNS = ["describe", "highlight", "tell me about", "what made",
                          "special about", "story of", "explain"]


def format_date(value) -> str:
    if value is None or pd.isna(value):
        return ""
    parsed = pd.to_datetime(value, errors="coerce")
    if not pd.isna(parsed):
        return parsed.strftime("%Y-%m-%d")
    return str(value)[:10]


def opponent_of(row, team: str) -> Optional[str]:
    team_n = normalize_team(team)
    t1 = clean_text(row.get("Team1 Name"))
    t2 = clean_text(row.get("Team2 Name"))
    if normalize_team(t1) == team_n:
        return t2
    if normalize_team(t2) == team_n:
        return t1
    return None


def winner_from_result_text(result_text: str) -> Optional[str]:
    if not result_text:
        return None
    text = clean_text(result_text)
    if not text:
        return None
    boundary_match = re.search(r"\(([^()]+?)\s+won\s+on\s+boundary\s+count\)", text, flags=re.IGNORECASE)
    if boundary_match:
        return boundary_match.group(1).strip()
    generic_match = re.search(r"^(.+?)\s+won\b", text, flags=re.IGNORECASE)
    if generic_match:
        return generic_match.group(1).strip()
    return None


_VENUE_STADIUM_COLUMNS = ["Match Venue (Stadium)", "Venue", "Stadium"]
_VENUE_CITY_COLUMNS = ["Match Venue (City)", "City"]


def _first_present(row, columns):
    for col in columns:
        if col in row.index:
            val = clean_text(row.get(col))
            if val:
                return val
    return ""


def exact_cricket_answer(filtered: pd.DataFrame, intent: str, question: str = "") -> Optional[str]:
    if filtered is None or filtered.empty:
        return None
    q = str(question).lower().strip()

    if len(filtered) != 1:
        if intent == "toss_winner":
            lines = []
            for _, row in filtered.iterrows():
                date = format_date(row.get("Match Date"))
                toss = clean_text(row.get("Toss Winner"))
                if toss:
                    lines.append(f"{date}: {toss} won the toss." if date else f"{toss} won the toss.")
            return "\n".join(lines) if lines else None

        if intent == "toss_decision":
            lines = []
            for _, row in filtered.iterrows():
                date = format_date(row.get("Match Date"))
                toss = clean_text(row.get("Toss Winner"))
                choice = clean_text(row.get("Toss Winner Choice"))
                if toss and choice:
                    cl = choice.lower()
                    decision = "chose to bat" if "bat" in cl else "chose to bowl" if "bowl" in cl \
                        else "chose to field" if "field" in cl else f"chose to {choice}"
                    lines.append(f"{date}: {toss} won the toss and {decision}.")
            return "\n".join(lines) if lines else None

        return None

    row = filtered.iloc[0]
    date = format_date(row.get("Match Date"))

    # Descriptive/open-ended questions should go to RAG, not this terse shortcut
    if (intent == "general") and any(p in q for p in _DESCRIPTIVE_PATTERNS):
        return None

    if intent == "toss_winner":
        toss = clean_text(row.get("Toss Winner"))
        if toss:
            return f"{toss} won the toss on {date}." if date else f"{toss} won the toss."
        return None

    if intent == "toss_decision":
        toss = clean_text(row.get("Toss Winner"))
        choice = clean_text(row.get("Toss Winner Choice"))
        if not toss or not choice:
            return None
        cl = choice.lower()
        decision = "chose to bat" if "bat" in cl else "chose to bowl" if "bowl" in cl \
            else "chose to field" if "field" in cl else f"chose to {choice}"
        return f"{toss} won the toss and {decision}."

    if intent == "winner":
        winner = clean_text(row.get("Match Winner"))
        if winner:
            return f"{winner} won the match."
        result = clean_text(row.get("Match Result Text"))
        result_winner = winner_from_result_text(result)
        if result_winner:
            if "boundary count" in result.lower():
                return f"The final was tied, but {result_winner} won on boundary count."
            return f"{result_winner} won the match."
        return None

    if intent == "team_runs":
        teams = extract_teams(question)
        if len(teams) == 1:
            requested_team = teams[0]
            requested_n = normalize_team(requested_team)
            team1 = clean_text(row.get("Team1 Name"))
            team2 = clean_text(row.get("Team2 Name"))
            if normalize_team(team1) == requested_n:
                runs = row.get("Team1 Runs Scored")
            elif normalize_team(team2) == requested_n:
                runs = row.get("Team2 Runs Scored")
            else:
                runs = None
            if pd.notna(runs):
                return f"{requested_team} scored {runs:g} runs."
            return None

        team1 = clean_text(row.get("Team1 Name"))
        team2 = clean_text(row.get("Team2 Name"))
        runs1 = row.get("Team1 Runs Scored")
        runs2 = row.get("Team2 Runs Scored")
        if pd.notna(runs1) and pd.notna(runs2):
            return f"{team1} scored {runs1:g} runs and {team2} scored {runs2:g} runs."
        return None

    if intent == "venue":
        stadium = _first_present(row, _VENUE_STADIUM_COLUMNS)
        city = _first_present(row, _VENUE_CITY_COLUMNS)
        if stadium and city:
            return f"The match was played at {stadium}, {city}."
        if stadium:
            return f"The match was played at {stadium}."
        return None

    if intent == "date":
        return f"The match was played on {date}." if date else None

    if intent == "final_teams":
        team1 = clean_text(row.get("Team1 Name"))
        team2 = clean_text(row.get("Team2 Name"))
        if team1 and team2:
            return f"The teams in the final were {team1} and {team2}."
        return None

    if intent == "final_opponent":
        teams = extract_teams(question)
        winner = clean_text(row.get("Match Winner"))
        team1 = clean_text(row.get("Team1 Name"))
        team2 = clean_text(row.get("Team2 Name"))
        if teams:
            requested_team = teams[0]
            opponent = opponent_of(row, requested_team)
            if opponent:
                if "lost to" in q:
                    return f"{requested_team} lost to {opponent} in the final."
                if winner:
                    return f"{winner} defeated {opponent} in the final."
                result = clean_text(row.get("Match Result Text"))
                result_winner = winner_from_result_text(result)
                if result_winner:
                    return f"{result_winner} defeated {opponent} in the final."
        if winner:
            if normalize_team(team1) == normalize_team(winner):
                opponent = team2
            elif normalize_team(team2) == normalize_team(winner):
                opponent = team1
            else:
                opponent = None
            if opponent:
                return f"{winner} defeated {opponent} in the final."
        return None

    if intent == "runner_up":
        winner = clean_text(row.get("Match Winner"))
        team1 = clean_text(row.get("Team1 Name"))
        team2 = clean_text(row.get("Team2 Name"))
        candidate_winner = winner
        if not candidate_winner:
            result = clean_text(row.get("Match Result Text"))
            candidate_winner = winner_from_result_text(result)
        if candidate_winner:
            if normalize_team(team1) == normalize_team(candidate_winner):
                runner_up = team2
            elif normalize_team(team2) == normalize_team(candidate_winner):
                runner_up = team1
            else:
                runner_up = None
            if runner_up:
                return f"{runner_up} was the runner-up."
        return None

    if intent == "general" or intent == "match_result":
        result = clean_text(row.get("Match Result Text"))
        if result:
            return result
        return None

    return None


def card_answer(question: str, intent: str) -> Optional[str]:
    """Glue: filter matches for the question, then apply the exact-answer layer."""
    filtered = metadata_search(question)
    return exact_cricket_answer(filtered, intent, question)


print("✅ Exact metadata answer layer loaded.")


✅ Exact metadata answer layer loaded.


## Match summary, Player of the Match, and scoped player/aggregate statistics (runs, wickets, top scorer, top wicket-taker — with optional opponent/year/final scoping)

In [12]:

def _format_score(runs, wickets):
    try:
        runs = float(runs)
        if pd.isna(runs):
            return None
        if wickets is not None:
            wickets = float(wickets)
            if not pd.isna(wickets):
                return f"{runs:g}/{wickets:g}"
        return f"{runs:g}"
    except Exception:
        return None


def _get_match_score_summary(row):
    if row is None:
        return None
    team1 = _safe_text(row.get("Team1 Name"))
    team2 = _safe_text(row.get("Team2 Name"))
    team1_score = _format_score(row.get("Team1 Runs Scored"), row.get("Team1 Wickets Fell"))
    team2_score = _format_score(row.get("Team2 Runs Scored"), row.get("Team2 Wickets Fell"))
    parts = []
    if team1 and team1_score:
        parts.append(f"{team1}: {team1_score}")
    if team2 and team2_score:
        parts.append(f"{team2}: {team2_score}")
    return " | ".join(parts) if parts else None


def _get_mom_name(row):
    if row is None:
        return None
    mom_value = row.get("MOM Player")
    if mom_value is None:
        return None
    player_name = _get_player_name_by_id(mom_value)
    if player_name:
        return player_name
    mom_text = _safe_text(mom_value)
    if not mom_text:
        return None
    try:
        numeric_value = pd.to_numeric(mom_text, errors="coerce")
        if pd.notna(numeric_value):
            return None
    except Exception:
        pass
    return mom_text


def _match_summary_answer(question):
    row = _find_match_for_question(question)
    if row is None:
        return None
    parts = []
    team1 = _safe_text(row.get("Team1 Name"))
    team2 = _safe_text(row.get("Team2 Name"))
    if team1 and team2:
        parts.append(f"{team1} played {team2}.")
    match_date = format_date(row.get("Match Date"))
    if match_date:
        parts.append(f"The match was played on {match_date}.")
    score_summary = _get_match_score_summary(row)
    if score_summary:
        parts.append(f"Final scores: {score_summary}.")
    result_text = _safe_text(row.get("Match Result Text"))
    winner = _safe_text(row.get("Match Winner"))
    if result_text:
        parts.append(result_text.rstrip(".") + ".")
    elif winner:
        parts.append(f"{winner} won the match.")
    mom = _get_mom_name(row)
    if mom:
        parts.append(f"Player of the Match: {mom}.")
    return " ".join(parts) if parts else None


def player_of_match_answer(question):
    row = _find_match_for_question(question)
    if row is None:
        return None
    mom = _get_mom_name(row)
    if not mom:
        return None
    return f"{mom} was the Player of the Match."


def _is_toss_decision_question(question):
    q = _safe_text(question).lower()
    patterns = [
        "what did they choose", "what did they decide", "what did the team choose",
        "what did the team decide", "choose after winning the toss", "decide after winning the toss",
        "what was their decision", "what did they elect", "elected to bat", "elected to bowl",
        "chose to bat", "chose to bowl",
    ]
    return any(p in q for p in patterns)


def _scope_text_for(question):
    opponent = _extract_opponent_from_question(question)
    year = extract_year(question)
    if is_final_question(question):
        return "in the final"
    if opponent and year:
        return f"against {opponent} in {year}"
    if opponent:
        return f"against {opponent}"
    if year:
        return f"in {year}"
    return "in the available ODI data"


def _get_all_batting_for_player(player_id):
    if batting is None or batting.empty or "batsman" not in batting.columns:
        return pd.DataFrame()
    pid = _normalize_id(player_id)
    if pid is None:
        return pd.DataFrame()
    ids = batting["batsman"].map(_normalize_id)
    df = batting[ids.eq(pid)].copy()
    if "runs" in df.columns:
        df["runs"] = pd.to_numeric(df["runs"], errors="coerce")
    return df


def _get_all_bowling_for_player(player_id):
    if bowling is None or bowling.empty or "bowler id" not in bowling.columns:
        return pd.DataFrame()
    pid = _normalize_id(player_id)
    if pid is None:
        return pd.DataFrame()
    ids = bowling["bowler id"].map(_normalize_id)
    df = bowling[ids.eq(pid)].copy()
    if "wickets" in df.columns:
        df["wickets"] = pd.to_numeric(df["wickets"], errors="coerce")
    return df


def _apply_match_scope(df, question):
    """Restrict a Match-ID-indexed dataframe to the scope implied by the question, if any."""
    scoped_ids = _get_scoped_match_ids(question)
    if scoped_ids is None:
        return df
    if "Match ID" not in df.columns:
        return df.iloc[0:0]
    mids = df["Match ID"].map(_normalize_id)
    return df[mids.isin(scoped_ids)].copy()


def deterministic_player_runs(question):
    player_id = _find_player_id_from_question(question)
    if player_id is None:
        return None
    player_name = _get_player_name_by_id(player_id)
    if not player_name:
        return None
    df = _get_all_batting_for_player(player_id)
    if df.empty:
        return None
    df = _apply_match_scope(df, question)
    if df.empty or "runs" not in df.columns:
        return None
    df = df[df["runs"].notna()]
    if df.empty:
        return None
    total_runs = df["runs"].sum()
    return f"{player_name} scored {float(total_runs):g} runs {_scope_text_for(question)}."


def deterministic_player_wickets(question):
    player_id = _find_player_id_from_question(question)
    if player_id is None:
        return None
    player_name = _get_player_name_by_id(player_id)
    if not player_name:
        return None
    df = _get_all_bowling_for_player(player_id)
    if df.empty:
        return None
    df = _apply_match_scope(df, question)
    if df.empty or "wickets" not in df.columns:
        return None
    df = df[df["wickets"].notna()]
    if df.empty:
        return None
    total_wickets = df["wickets"].sum()
    return f"{player_name} took {float(total_wickets):g} wickets {_scope_text_for(question)}."


def _top_from_grouped(grouped):
    if grouped.empty:
        return None, []
    max_value = grouped.iloc[0]
    top_ids = grouped[grouped.eq(max_value)].index.tolist()
    names = []
    for pid in top_ids:
        name = _get_player_name_by_id(pid)
        if name:
            names.append(name)
    names = list(dict.fromkeys(names))
    return max_value, names

# ============================================================
# FOW (fall of wickets) — deterministic handler
# Place this in a NEW cell directly AFTER the cell that defines
# _STRUCTURED_HANDLERS (Cell 24). Do not edit any other cell.
# ============================================================

_WICKET_WORDS = {
    "first": 1, "1st": 1, "second": 2, "2nd": 2, "third": 3, "3rd": 3,
    "fourth": 4, "4th": 4, "fifth": 5, "5th": 5, "sixth": 6, "6th": 6,
    "seventh": 7, "7th": 7, "eighth": 8, "8th": 8, "ninth": 9, "9th": 9,
    "tenth": 10, "10th": 10,
}


def _extract_wicket_number(question: str):
    """Return an int (1-10), the string 'last', or None (= no specific wicket asked)."""
    q = clean_text(question).lower()

    if "last wicket" in q:
        return "last"

    for word, number in _WICKET_WORDS.items():
        if re.search(rf"\b{re.escape(word)}\s+wicket", q):
            return number

    m = re.search(r"\bwicket\s*(?:number|no\.?|#)?\s*(\d{1,2})\b", q)
    if m and 1 <= int(m.group(1)) <= 10:
        return int(m.group(1))

    return None


def _extract_innings_number(question: str):
    q = clean_text(question).lower()
    if re.search(r"\b(?:first|1st)\s+innings\b", q):
        return 1
    if re.search(r"\b(?:second|2nd)\s+innings\b", q):
        return 2
    return None


def _fow_player_name(player_id) -> str:
    name = _get_player_name_by_id(player_id)
    if name:
        return name
    pid = _normalize_id(player_id)
    return f"player {pid}" if pid else "an unknown batter"


def _fow_row_text(row) -> str:
    wicket = int(float(row["wicket"]))
    name = _fow_player_name(row.get("player"))

    runs = row.get("runs")
    score = f"{float(runs):g}/{wicket}" if pd.notna(runs) else "score not recorded"

    over = row.get("over")
    over_text = f" in over {float(over):g}" if "over" in row.index and pd.notna(over) else ""

    return f"wicket {wicket}: {name} was dismissed at {score}{over_text}"


def deterministic_fow(question):
    """
    Answer fall-of-wickets questions straight from the `fow` table.

    - Match is resolved with the existing _find_match_for_question().
    - Wicket number comes from the question ("first", "3rd", "last", ...).
    - If a team / innings is named, only that innings is used; otherwise both innings.
    - If no wicket number is given, the full fall of wickets is returned.
    """
    if fow is None or fow.empty:
        return None
    if not {"Match ID", "innings", "team", "player", "wicket", "runs"}.issubset(fow.columns):
        return None

    match = _find_match_for_question(question)
    if match is None:
        return None

    match_id = _normalize_id(match.get("Match ID"))
    if match_id is None:
        return None

    rows = fow[fow["Match ID"].map(_normalize_id).eq(match_id)].copy()
    if rows.empty:
        return None

    rows["innings"] = pd.to_numeric(rows["innings"], errors="coerce")
    rows["wicket"] = pd.to_numeric(rows["wicket"], errors="coerce")
    rows = rows.dropna(subset=["innings", "wicket"])
    if rows.empty:
        return None

    # --- optional narrowing: innings number, then a single named team -----------
    innings_wanted = _extract_innings_number(question)
    if innings_wanted is not None:
        rows = rows[rows["innings"].eq(innings_wanted)]
    else:
        named = extract_teams(question)
        if len(named) == 1:
            named_n = _normalize_entity_for_match(named[0])
            team_rows = rows[rows["team"].map(_normalize_entity_for_match).eq(named_n)]
            if not team_rows.empty:
                rows = team_rows
    if rows.empty:
        return None

    wicket_wanted = _extract_wicket_number(question)

    t1 = clean_text(match.get("Team1 Name"))
    t2 = clean_text(match.get("Team2 Name"))
    date = format_date(match.get("Match Date"))
    header = f"{t1} vs {t2}" + (f" ({date})" if date else "")

    lines = []
    for innings_no in sorted(rows["innings"].unique()):
        inn = rows[rows["innings"].eq(innings_no)].sort_values("wicket", kind="stable")
        team = clean_text(inn["team"].iloc[0])

        if wicket_wanted is None:
            parts = [
                f"{int(w)}-{float(r):g} ({_fow_player_name(p)})" if pd.notna(r)
                else f"{int(w)}-? ({_fow_player_name(p)})"
                for w, r, p in zip(inn["wicket"], inn["runs"], inn["player"])
            ]
            lines.append(f"{team} innings — fall of wickets: " + ", ".join(parts))
            continue

        target = int(inn["wicket"].max()) if wicket_wanted == "last" else wicket_wanted
        hit = inn[inn["wicket"].eq(target)]
        if hit.empty:
            lines.append(f"{team} innings — no wicket number {target} recorded.")
        else:
            lines.append(f"{team} innings — {_fow_row_text(hit.iloc[0])}.")

    if not lines:
        return None
    return header + "\n" + "\n".join(lines)


def deterministic_highest_score(question):
    """
    Return the highest individual score in the match/scope.

    Important:
        Unlike top_scorer, this does NOT sum a player's runs
        across multiple rows/matches.

        It finds the maximum single-innings score from the
        batting dataframe within the question's match scope.
    """

    if batting is None or batting.empty:
        return None

    if "batsman" not in batting.columns or "runs" not in batting.columns:
        return None

    df = batting.copy()

    # Make sure runs are numeric
    df["runs"] = pd.to_numeric(df["runs"], errors="coerce")
    df = df[df["runs"].notna()]

    if df.empty:
        return None

    # Restrict to the match/year/final mentioned in the question
    df = _apply_match_scope(df, question)

    if df.empty:
        return None

    # Highest SINGLE innings score
    max_runs = df["runs"].max()

    if pd.isna(max_runs):
        return None

    # There can be multiple players tied on the same highest score
    top_rows = df[df["runs"].eq(max_runs)]

    names = []

    for player_id in top_rows["batsman"].dropna().unique():
        name = _get_player_name_by_id(player_id)

        if name:
            names.append(name)

    names = list(dict.fromkeys(names))

    if not names:
        return None

    scope = _scope_text_for(question)

    if len(names) == 1:
        return (
            f"{names[0]} scored the highest individual score "
            f"with {float(max_runs):g} runs {scope}."
        )

    return (
        f"{', '.join(names)} jointly had the highest individual score "
        f"with {float(max_runs):g} runs each {scope}."
    )


def deterministic_top_wicket_taker(question):
    if bowling is None or bowling.empty or "bowler id" not in bowling.columns or "wickets" not in bowling.columns:
        return None
    df = bowling.copy()
    df["wickets"] = pd.to_numeric(df["wickets"], errors="coerce")
    df = df[df["wickets"].notna()]
    df = _apply_match_scope(df, question)
    if df.empty:
        return None
    grouped = df.groupby("bowler id", dropna=True)["wickets"].sum().sort_values(ascending=False)
    max_wickets, names = _top_from_grouped(grouped)
    if not names:
        return None
    scope = _scope_text_for(question)
    if len(names) == 1:
        return f"{names[0]} took {float(max_wickets):g} wickets, the most {scope}."
    return f"{', '.join(names)} took {float(max_wickets):g} wickets each, the most {scope}."

def deterministic_top_scorer(question):
    """Aggregate runs leaderboard (sum across matches/scope) — analogous to
    deterministic_top_wicket_taker but for batting. NOTE: unlike
    deterministic_highest_score, this sums a player's runs across every row
    in scope rather than looking at a single innings."""
    if batting is None or batting.empty or "batsman" not in batting.columns or "runs" not in batting.columns:
        return None
    df = batting.copy()
    df["runs"] = pd.to_numeric(df["runs"], errors="coerce")
    df = df[df["runs"].notna()]
    df = _apply_match_scope(df, question)
    if df.empty:
        return None
    grouped = df.groupby("batsman", dropna=True)["runs"].sum().sort_values(ascending=False)
    max_runs, names = _top_from_grouped(grouped)
    if not names:
        return None
    scope = _scope_text_for(question)
    if len(names) == 1:
        return f"{names[0]} scored {float(max_runs):g} runs, the most {scope}."
    return f"{', '.join(names)} scored {float(max_runs):g} runs each, the most {scope}."

print("✅ Structured statistics engine loaded.")


✅ Structured statistics engine loaded.


## Semantic retrieval (RAG) + grounded QLoRA generation — used only for open-ended/descriptive questions the structured layer can't answer

In [13]:
def semantic_search(query: str, top_k: int = 10) -> List[Dict]:
    if embeddings is None or documents is None or embedding_model is None:
        return []
    query_embedding = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    matrix = np.asarray(embeddings)
    if matrix.ndim != 2 or matrix.shape[0] != len(documents):
        return []
    scores = matrix @ query_embedding
    indices = np.argsort(scores)[::-1][:top_k]
    return [{"index": int(i), "score": float(scores[i]), "document": str(documents[i])} for i in indices]


def route_dataset(intent):
    if intent in {"winner", "result", "venue", "date", "toss_winner", "toss_decision", "team_runs",
                  "highest_team_runs", "player_of_match", "captain", "general", "match_summary",
                  "match_result", "final_teams", "final_opponent", "runner_up"}:
        return "matches"
    if intent in {"top_scorer", "highest_score", "player_runs", "batting_card"}:
        return "batting"
    if intent in {"top_wicket_taker", "player_wickets", "bowling_card", "wickets"}:
        return "bowling"
    if intent in {"fow"}:
        return "fow"
    if intent in {"partnership"}:
        return "partnership"
    return "matches"


def enrich_with_names_and_year(df: pd.DataFrame) -> pd.DataFrame:
    """Join player names (from `players`) and match year + match name
    (from `matches`) into a stats table that only has numeric IDs."""
    df = df.copy()

    # Attach match year AND match name (needed to detect "final" matches)
    if "Match ID" in df.columns and "Match ID" in matches.columns:
        df = df.merge(matches[["Match ID", "_year", "Match Name"]], on="Match ID", how="left")

    # Attach player name for any *_id column that refers to a player
    id_cols = [c for c in df.columns if c.lower() in
               ("bowler id", "batsman id", "batter id", "player id", "player1", "player2")]
    for col in id_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        lookup = players[["player_id", "player_name"]].copy()
        lookup["player_id"] = pd.to_numeric(lookup["player_id"], errors="coerce")
        df = df.merge(lookup, left_on=col, right_on="player_id", how="left")
        df = df.rename(columns={"player_name": col.replace(" id", " name").replace("player", "player_name")})
        df = df.drop(columns=["player_id"], errors="ignore")

    return df


def filter_card_dataset(df: pd.DataFrame, question: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    filtered = enrich_with_names_and_year(df)
    q = clean_text(question).lower()

    # Filter by year
    year = extract_year(q)
    if year is not None and "_year" in filtered.columns:
        year_mask = filtered["_year"].eq(float(year))
        if year_mask.any():
            filtered = filtered[year_mask]

    # NEW: narrow to "final" matches when the question says "final"
    if "final" in q and "Match Name" in filtered.columns:
        final_mask = filtered["Match Name"].astype(str).str.lower().str.contains("final", na=False)
        if final_mask.any():
            filtered = filtered[final_mask]

    # Filter by player name
    name_cols = [c for c in filtered.columns if "name" in c.lower() and "team" not in c.lower() and c != "Match Name"]
    for col in name_cols:
        mask = filtered[col].astype(str).str.lower().apply(
            lambda name: (name in q) if name and name != "nan" else False
        )
        if mask.any():
            filtered = filtered[mask]
            break

    return filtered

print("✅ filter_card_dataset now also narrows by 'final'.")

def build_context(question: str, top_k: int = 5) -> Tuple[str, List[Dict]]:
    intent = detect_intent(question)
    route = route_dataset(intent)
    contexts, sources = [], []

    if route == "matches":
        df = metadata_search(question)
        if not df.empty:
            for idx, row in df.head(top_k).iterrows():
                text = " | ".join(
                    f"{col}: {clean_text(row.get(col))}"
                    for col in df.columns if not str(col).startswith("_") and clean_text(row.get(col))
                )
                contexts.append(text)
                sources.append({"source": "matches", "index": int(idx), "date": format_date(row.get("Match Date"))})
    elif route in {"bowling", "fow", "partnership"}:
        source_df = {"bowling": bowling, "fow": fow, "partnership": partnership}[route]
        filtered = filter_card_dataset(source_df, question)
        for idx, row in filtered.head(top_k).iterrows():
            text = " | ".join(
                f"{col}: {clean_text(row.get(col))}"
                for col in filtered.columns if not str(col).startswith("_") and clean_text(row.get(col))
            )
            contexts.append(text)
            sources.append({"source": route, "index": int(idx)})
    elif route == "batting":
        filtered = filter_card_dataset(batting, question)
        for idx, row in filtered.head(top_k).iterrows():
            text = " | ".join(
                f"{col}: {clean_text(row.get(col))}"
                for col in filtered.columns if not str(col).startswith("_") and clean_text(row.get(col))
            )
            contexts.append(text)
            sources.append({"source": "batting", "index": int(idx)})

    if not contexts:
        semantic = semantic_search(question, top_k=top_k)
        for item in semantic:
            contexts.append(item["document"])
            sources.append({"source": "semantic", "index": item["index"], "score": item["score"]})

    return "\n\n".join(contexts), sources


def rag_answer(question: str, context: str) -> str:
    """
    Generate a grounded answer using only the retrieved cricket context.

    The model should:
    - use only information present in DATA
    - answer normally when relevant information exists
    - avoid refusing just because some unrelated field is missing
    - never invent facts
    """

    fallback = "I could not find that information in the available cricket data."

    if not context or not context.strip():
        return fallback

    prompt = f"""
You are a precise cricket data question-answering system.

Answer the QUESTION using ONLY the information explicitly present in DATA.

Rules:
1. Do not use outside cricket knowledge.
2. Do not guess or invent facts.
3. If DATA contains relevant information, use it to answer the question.
4. You do NOT need to mention every field in DATA.
5. Do NOT refuse just because some information is missing.
6. Keep the answer clear and concise.
7. For questions asking what made a match special, describe the important
   facts present in DATA that explain why the match was notable.
8. If DATA contains no useful information for answering the QUESTION,
   return exactly:
   "{fallback}"

DATA:
{context}

QUESTION:
{question}

ANSWER:
""".strip()

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(generator.device)

    import torch

    with torch.no_grad():
        outputs = generator.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return answer if answer else fallback

✅ filter_card_dataset now also narrows by 'final'.


## Master `answer_question()` — the single entry point

In [14]:
def deterministic_loser(question):
    """
    Return the losing team for the match resolved from the question.

    Normal case:
        Use Match Winner.

    Special case:
        If Match Winner is missing/NaN, inspect Match Result Text.
        This handles matches such as the 2019 World Cup Final where
        the match was tied but England won on boundary count.
    """

    row = _find_match_for_question(question)

    if row is None:
        return None

    team1 = clean_text(row.get("Team1 Name", ""))
    team2 = clean_text(row.get("Team2 Name", ""))

    if not team1 or not team2:
        return None

    # ---------------------------------------------------------
    # 1. Normal case: Match Winner is available
    # ---------------------------------------------------------

    winner_raw = row.get("Match Winner", "")

    # Handle pandas NaN / None safely
    try:
        if pd.isna(winner_raw):
            winner_raw = ""
    except Exception:
        pass

    winner = clean_text(winner_raw)

    if winner:
        winner_n = _normalize_entity_for_match(winner)
        team1_n = _normalize_entity_for_match(team1)
        team2_n = _normalize_entity_for_match(team2)

        if winner_n == team1_n:
            return f"{team2} lost the match."

        if winner_n == team2_n:
            return f"{team1} lost the match."

    # ---------------------------------------------------------
    # 2. Special case: Match Winner is missing
    #    Use Match Result Text
    # ---------------------------------------------------------

    result_text = clean_text(row.get("Match Result Text", ""))

    if result_text:
        result_n = result_text.lower()

        team1_n = _normalize_entity_for_match(team1)
        team2_n = _normalize_entity_for_match(team2)

        # Check whether Team 1 is explicitly mentioned as winner
        if team1_n and team1_n.lower() in result_n:
            if "won" in result_n:
                return f"{team2} lost the match."

        # Check whether Team 2 is explicitly mentioned as winner
        if team2_n and team2_n.lower() in result_n:
            if "won" in result_n:
                return f"{team1} lost the match."

    # ---------------------------------------------------------
    # 3. Could not determine a reliable loser
    # ---------------------------------------------------------

    return None
_NUMERIC_CRITICAL_INTENTS = {
    "player_runs", "player_wickets", "top_scorer", "top_wicket_taker",
    "team_runs", "highest_team_runs", "winner", "toss_winner", "toss_decision",
    "player_of_match", "runner_up", "final_teams", "final_opponent", "venue", "date", "loser",
    "fow", "wickets",  # unsupported FOW questions => "no reliable answer", not a RAG guess
}

_STRUCTURED_HANDLERS = {
    "player_runs": deterministic_player_runs,
    "player_wickets": deterministic_player_wickets,
    "top_scorer": deterministic_top_scorer,
    "highest_score": deterministic_highest_score,
    "top_wicket_taker": deterministic_top_wicket_taker,
    "player_of_match": player_of_match_answer,
    "loser": deterministic_loser,
    "fow": deterministic_fow,
}


def answer_question(question: str) -> str:
    question = clean_text(question)

    if not question:
        return "Please enter a cricket question."

    if not is_cricket_question(question):
        return "I can answer cricket-related questions from the available cricket data."

    intent = detect_intent(question)

    if _is_toss_decision_question(question):
        intent = "toss_decision"

    handler = _STRUCTURED_HANDLERS.get(intent)

    if handler:
        try:
            answer = handler(question)
        except Exception:
            answer = None

        if answer:
            return answer

    if intent == "match_summary":
        try:
            summary = _match_summary_answer(question)
        except Exception:
            summary = None

        if summary:
            return summary
    try:
        exact_answer = card_answer(question, intent)
    except Exception:
        exact_answer = None

    if exact_answer:
        exact_answer = clean_text(exact_answer)

        if exact_answer:
            return exact_answer

    if intent == "general":
        try:
            summary = _match_summary_answer(question)
        except Exception:
            summary = None

        if summary:
            return summary

    if intent in _NUMERIC_CRITICAL_INTENTS:
        return "I could not find a reliable answer in the available cricket data."

    try:
        context, _sources = build_context(question, top_k=8)
    except Exception:
        context = ""

    if not context.strip():
        return "I could not find a reliable answer in the available cricket data."

    try:
        answer = rag_answer(question, context)
    except Exception:
        return "I could not generate a reliable answer from the available cricket data."

    answer = clean_text(answer)

    return answer or "I could not find a reliable answer in the available cricket data."


print("✅ Master answer engine loaded — ready for QA.")

✅ Master answer engine loaded — ready for QA.


In [15]:
test_cases = [
    "who won the 2023 world cup final",
    "who won the 2023 world cup qualifier final",
    "who was player of the match in the 2023 world cup final",
    "who won the 2023 world cup",  # sanity check - should still resolve fine
]

for q in test_cases:
    print(f"Q: {q}")
    try:
        answer = answer_question(q)
    except Exception as e:
        answer = f"❌ ERROR: {e}"
    print(f"A: {answer}")
    print("-" * 60)

Q: who won the 2023 world cup final
A: Australia won the match.
------------------------------------------------------------
Q: who won the 2023 world cup qualifier final
A: Sri Lanka won the match.
------------------------------------------------------------
Q: who was player of the match in the 2023 world cup final
A: Travis Head was the Player of the Match.
------------------------------------------------------------
Q: who won the 2023 world cup
A: Australia won the match.
------------------------------------------------------------


In [16]:
regression_tests = [
    "Who was the first wicket to fall in the 1999 World Cup Final?",   # different final, sanity check
    "Fall of wickets for Pakistan in the 2017 Champions Trophy final", # non-World-Cup "final" — should this resolve at all?
    "How many runs had India scored when they lost their 5th wicket in the 2003 World Cup Final?",
    "Who was the top scorer in the 2019 World Cup Final?",             # unrelated intent — make sure it's untouched
    "What was the fall of wickets in a random 2016 T20 league match?", # no World Cup/final context, just a normal match
]
for q in regression_tests:
    print("Q:", q)
    print(answer_question(q))
    print("-" * 60)

Q: Who was the first wicket to fall in the 1999 World Cup Final?
Pakistan vs Australia (1999-06-20)
Pakistan innings — wicket 1: Wajahatullah Wasti was dismissed at 21/1 in over 4.4.
Australia innings — wicket 1: Adam Gilchrist was dismissed at 75/1 in over 10.1.
------------------------------------------------------------
Q: Fall of wickets for Pakistan in the 2017 Champions Trophy final
Pakistan vs India (2017-06-18)
Pakistan innings — fall of wickets: 1-128 (Azhar Ali), 2-200 (Fakhar Zaman), 3-247 (Shoaib Malik), 4-267 (Babar Azam)
------------------------------------------------------------
Q: How many runs had India scored when they lost their 5th wicket in the 2003 World Cup Final?
Australia vs India (2003-03-23)
India innings — wicket 5: Rahul Dravid was dismissed at 187/5 in over 31.5.
------------------------------------------------------------
Q: Who was the top scorer in the 2019 World Cup Final?
Ben Stokes scored 84 runs, the most in the final.
-----------------------------

## Chat interface

In [17]:
# ============================================
# CRICKET AI CHATBOT UI
# ============================================
import gradio as gr

def cricket_chat(message, history):
    if not message or not message.strip():
        return "Please enter a cricket question."

    try:
        return answer_question(message)
    except Exception as e:
        print("[ERROR] Chatbot error:", repr(e))
        return "Sorry, I could not process your question."


theme = gr.themes.Soft(
    primary_hue="green",
    secondary_hue="emerald",
    neutral_hue="slate",
).set(
    block_border_width="2px",
    input_border_width="0px",
    button_border_width="2px",

    # ---- LIGHT MODE ----
    input_background_fill="#f9fafb",
    input_background_fill_focus="#f9fafb",
    input_background_fill_hover="#f9fafb",
    block_border_color="#4ade80",
    input_border_color="#4ade80",

    # ---- DARK MODE ----
    input_background_fill_dark="#111827",
    input_background_fill_focus_dark="#111827",
    input_background_fill_hover_dark="#111827",
    block_border_color_dark="#374151",
    input_border_color_dark="#374151",
)

custom_css = """
/* ============ SAB MODES: input ke upar wale wrappers ka grey border khatam ============ */
div:has(#cricket-input) {
    border: none !important;
    box-shadow: none !important;
}

/* ================= LIGHT MODE ================= */
/* sirf ek border: green (upar wala rule is se pehle aata hai, is liye ye jeetta hai) */
.form:has(#cricket-input),
div:has(> #cricket-input) {
    background: #f9fafb !important;
    border: 2px solid #4ade80 !important;
}

#cricket-input,
#cricket-input *:not(button):not(svg):not(path) {
    background: #f9fafb !important;
    border-color: transparent !important;
    box-shadow: none !important;
}

/* ================= DARK MODE ================= */
.dark .form:has(#cricket-input),
.dark div:has(> #cricket-input) {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

/* input ka dibba: chat box jaisa rang, patla border (1px) */
.dark #cricket-input {
    background: #111827 !important;
    border: 2px solid #374151 !important;
    box-shadow: none !important;
}

.dark #cricket-input *:not(button):not(svg):not(path) {
    background: transparent !important;
    border-color: transparent !important;
    box-shadow: none !important;
}
"""

chat_kwargs = dict(
    fn=cricket_chat,
    title="🏏 Cricket AI",
    description=(
        "Ask questions about cricket matches, players, scores, "
        "wickets, tosses and results."
    ),
    textbox=gr.Textbox(
        placeholder="Ask a cricket question...",
        label="Your Question",
        elem_id="cricket-input",
        submit_btn=True,
    ),
    examples=[
        "Who won the 2011 World Cup final?",
        "What was the score in the 2019 World Cup final?",
        "Who was the Player of the Match in the 1992 final?",
    ],
    cache_examples=False,
)

major = int(gr.__version__.split(".")[0])

if major >= 6:
    demo = gr.ChatInterface(**chat_kwargs)
    print("✅ Cricket AI chatbot created!")
    demo.launch(share=True, theme=theme, css=custom_css)
else:
    demo = gr.ChatInterface(theme=theme, css=custom_css, **chat_kwargs)
    print("✅ Cricket AI chatbot created!")
    demo.launch(share=True)

✅ Cricket AI chatbot created!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4eda497d3937c68948.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
